# Replicating the KLA submissionRe-derives every number in the README from the supplied data and the trainedcheckpoint. Nothing here depends on the original training run.**There are 9 steps. Each step is one heading followed by exactly one codecell, numbered the same.** Run them in order with `Shift+Enter`.**Do not use Run All** — read each result before moving on.| | needs GPU? | roughly ||---|---|---|| Steps 1–6 | no | 3 minutes || Steps 7–8 | **yes** | 6 minutes || Step 9 | no | seconds |---### Before you start1. **Add Input** → attach `kla-train` and `kla-test`. If the keyword search   does not find them (Kaggle is unreliable with your own private datasets),   paste the URL instead:   `https://www.kaggle.com/datasets/kushalmr9033/kla-train`2. **Settings → Internet → ON** — needed for `git clone`, `git-lfs` and the   LPIPS weights.3. **Settings → Accelerator → GPU T4 x2** — *not* P100: current PyTorch has   no kernels for its sm_60 architecture and every CUDA call fails.

## STEP 1 — fetch the code and find the dataExpect to see `best.pt` at roughly **108 MB** and file counts of**3200 / 3200 / 400**. If `best.pt` is under 1 MB, Git LFS did not fetch andthe cell stops with instructions.

In [ ]:
import os, sys, glob, json, subprocess, timeREPO = "https://github.com/Kushal-MR/kla-image-restoration.git"ROOT = "/kaggle/working/repo"# git-lfs must exist BEFORE cloning, or best.pt arrives as a tiny pointer fileif subprocess.run(["which", "git-lfs"], capture_output=True).returncode != 0:    print("installing git-lfs ...")    subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=False)subprocess.run(["git", "lfs", "install"], check=False, capture_output=True)if os.path.isdir(os.path.join(ROOT, ".git")):    subprocess.run(["git","-C",ROOT,"fetch","--quiet","origin"], check=True)    subprocess.run(["git","-C",ROOT,"reset","--hard","--quiet","origin/main"], check=True)else:    subprocess.run(["git","clone","--quiet",REPO,ROOT], check=True)subprocess.run(["git","-C",ROOT,"lfs","pull"], check=False)print("commit:", subprocess.run(["git","-C",ROOT,"log","-1","--format=%h %s"],                                capture_output=True, text=True).stdout.strip())ckpt = os.path.join(ROOT, "weights", "best.pt")mb = os.path.getsize(ckpt) / 1e6print(f"weights/best.pt: {mb:.1f} MB")assert mb > 50, f"best.pt is a Git LFS pointer. Run: !cd {ROOT} && git lfs install && git lfs pull"# train/GT and train/NoisyLR sit side by side. The TEST folder is ALSO called# NoisyLR, so find the training one as GT's sibling and the test one as the# other. Getting this wrong would silently score against the wrong data.gt_dirs = [d for d in glob.glob("/kaggle/input/**/GT", recursive=True) if os.path.isdir(d)]assert gt_dirs, "no GT folder found -- is kla-train attached?"GT = gt_dirs[0]LRD = os.path.join(os.path.dirname(GT), "NoisyLR")assert os.path.isdir(LRD), f"expected {LRD} next to {GT}"others = [d for d in glob.glob("/kaggle/input/**/NoisyLR", recursive=True)          if os.path.isdir(d) and os.path.abspath(d) != os.path.abspath(LRD)]TEST = others[0] if others else Nonen = lambda d: len(glob.glob(os.path.join(d, "*.npy"))) if d else 0print(f"train GT      : {GT}  ({n(GT)} files)")print(f"train NoisyLR : {LRD}  ({n(LRD)} files)")print(f"test  NoisyLR : {TEST}  ({n(TEST)} files)")assert n(GT) == n(LRD) > 0, "GT and NoisyLR counts disagree"if TEST is None:    print("\nNOTE: kla-test not attached. Steps 8 and 9 will not run;"          "\n      everything else still works.")

## STEP 2 — repository self-checkVerifies the checkpoint matches the architecture, the outputs are valid andthe README's numbers match `metrics.json`. Should end **ALL CHECKS PASSED**.

In [ ]:
!cd {ROOT} && python scripts/preflight.py

## STEP 3 — what is actually in the dataExpect: ground truth exactly in [0,1], and roughly **57% of degraded imagescontaining negative pixels** — the evidence that additive noise is present.

In [ ]:
!cd {ROOT} && python scripts/inspect_npy.py --gt_dir {GT} --lr_dir {LRD} --n 300

## STEP 4 — recover the degradation operator, rebuild the splitExpect **L near 37**, and the last line to read`val_hard reproduces configs/split.json: True`.That line is the important one. If it says `False`, stop — the validation setwould differ from the one the reported metrics were computed on, and nothingbelow would be comparable.Takes about 2 minutes.

In [ ]:
!cd {ROOT} && python scripts/day1_setup.py --gt_dir {GT} --lr_dir {LRD} \    --n 250 --out_split /kaggle/working/split_regen.json \    --compare_to configs/split.json

## STEP 5 — does our synthetic damage match KLA's real damage?The REAL and SYNTHETIC rows should agree closely on max, min, mean, std andthe percentage going negative. This is what justifies training on syntheticpairs at all.

In [ ]:
!cd {ROOT} && python src/make_training_data.py --compare_real --gt_dir {GT} --lr_dir {LRD}

## STEP 6 — install LPIPSOne of the three scored metrics. `report_results.py` runs without it, butthen LPIPS is missing from the results.

In [ ]:
try:    import lpips; print("lpips already available")except ImportError:    !pip install -q lpips    import lpips; print("lpips installed")

## STEP 7 — score the model  *(needs GPU)*Regenerates `metrics.json` and the six figures. Takes 3–5 minutes.Expect **PSNR ≈ 26.7 dB, SSIM ≈ 0.686, LPIPS ≈ 0.375**, against a bicubicbaseline near 22.9 / 0.541 / 0.448.If PSNR differs from 26.68 by more than about 0.1 dB, send me the numbersbefore submitting — it would mean two evaluation harnesses disagree.

In [ ]:
!cd {ROOT} && python report_results.py --gt_dir {GT} --lr_dir {LRD} \    --split configs/split.json --ckpt weights/best.pt \    --out /kaggle/working/results

## STEP 8 — time the inference script end to end  *(needs GPU)*This is the figure KLA measures: process start, imports, model load, diskread, inference, disk write — **not** the forward pass, which is only about aquarter of it.Whatever this prints is the number that belongs in the README.

In [ ]:
assert TEST, "kla-test is not attached -- skip this step"t0 = time.time()!cd {ROOT} && python inference.py {TEST} /kaggle/working/test_outprint(f"\nwall clock around the whole process: {time.time()-t0:.1f} s")

## STEP 9 — check the restored outputs are validEvery line should confirm: 400 files, all 256×256, range inside [0,1], nonon-finite values, filenames identical to the inputs.

In [ ]:
import numpy as npassert TEST, "kla-test is not attached -- skip this step"out = sorted(glob.glob("/kaggle/working/test_out/*.npy"))src = sorted(glob.glob(os.path.join(TEST, "*.npy")))shapes = {np.load(f).shape for f in out}mn = min(float(np.load(f).min()) for f in out)mx = max(float(np.load(f).max()) for f in out)bad = sum(0 if np.isfinite(np.load(f)).all() else 1 for f in out)print(f"restored {len(out)} of {len(src)} inputs")print(f"shapes            : {shapes}")print(f"value range       : [{mn:.4f}, {mx:.4f}]")print(f"non-finite files  : {bad}")print(f"filenames match   : "      f"{[os.path.basename(f) for f in out] == [os.path.basename(f) for f in src]}")